# 🌍 Multi-Source Health Data Comparison

This notebook demonstrates how to combine data from multiple sources for comprehensive health analysis.

**Data Sources Combined:**
- PySUS (Brazil - DATASUS)
- WHO Global Health Observatory
- World Bank Health Indicators
- ECDC European Surveillance

**Use Cases:**
- Cross-validate data across sources
- Compare health indicators across regions
- Identify data gaps and discrepancies
- Create comprehensive health dashboards

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

print("✅ Imports completed!")
print(f"⏰ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 2. Loading Data from Multiple Sources

In [ ]:
# Load data from previous notebooks (or re-download)
import os

output_dir = "./output"

# Try to load existing data
data_sources = {}

# WHO data
who_file = os.path.join(output_dir, 'who_health_comparison.csv')
if os.path.exists(who_file):
    data_sources['WHO'] = pd.read_csv(who_file, index_col=0)
    print("✅ Loaded WHO data")
else:
    print("⚠️ WHO data not found - run notebook 02 first")

# World Bank data
wb_file = os.path.join(output_dir, 'wb_health_dashboard.csv')
if os.path.exists(wb_file):
    data_sources['World Bank'] = pd.read_csv(wb_file, index_col=0)
    print("✅ Loaded World Bank data")
else:
    print("⚠️ World Bank data not found - run notebook 03 first")

print(f"\n📊 Available data sources: {len(data_sources)}")
for source, data in data_sources.items():
    print(f"  • {source}: {len(data)} countries, {len(data.columns)} indicators")

## 3. Life Expectancy Comparison Across Sources

In [ ]:
# Compare life expectancy data from different sources
print("📊 Comparing Life Expectancy Data Across Sources\n")

life_exp_comparison = pd.DataFrame()

# WHO Life Expectancy
if 'WHO' in data_sources and 'Life Expectancy' in data_sources['WHO'].columns:
    life_exp_comparison['WHO'] = data_sources['WHO']['Life Expectancy']

# World Bank Life Expectancy
if 'World Bank' in data_sources and 'Life Expectancy' in data_sources['World Bank'].columns:
    life_exp_comparison['World Bank'] = data_sources['World Bank']['Life Expectancy']

if not life_exp_comparison.empty:
    print("📋 Life Expectancy Comparison:")
    display(life_exp_comparison.round(2))
    
    # Calculate differences
    if len(life_exp_comparison.columns) > 1:
        life_exp_comparison['Difference'] = (
            life_exp_comparison.iloc[:, 0] - life_exp_comparison.iloc[:, 1]
        )
        print("\n📊 Differences between sources:")
        print(f"  Mean absolute difference: {life_exp_comparison['Difference'].abs().mean():.2f} years")
        print(f"  Max difference: {life_exp_comparison['Difference'].abs().max():.2f} years")

In [ ]:
# Visualize comparison
if not life_exp_comparison.empty and len(life_exp_comparison.columns) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Scatter plot comparison
    x = life_exp_comparison.iloc[:, 0]
    y = life_exp_comparison.iloc[:, 1]
    
    axes[0].scatter(x, y, s=100, alpha=0.7, edgecolors='black')
    
    # Add diagonal line (perfect agreement)
    min_val = min(x.min(), y.min())
    max_val = max(x.max(), y.max())
    axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5, label='Perfect agreement')
    
    # Add country labels
    for country in life_exp_comparison.index:
        if pd.notna(x[country]) and pd.notna(y[country]):
            axes[0].annotate(country, (x[country], y[country]),
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    axes[0].set_xlabel(f'{life_exp_comparison.columns[0]} (years)')
    axes[0].set_ylabel(f'{life_exp_comparison.columns[1]} (years)')
    axes[0].set_title('Life Expectancy: Source Comparison')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Calculate correlation
    corr = x.corr(y)
    axes[0].text(0.05, 0.95, f'Correlation: {corr:.3f}',
                transform=axes[0].transAxes, bbox=dict(boxstyle='round', facecolor='wheat'))
    
    # Bar chart of differences
    if 'Difference' in life_exp_comparison.columns:
        diff_sorted = life_exp_comparison['Difference'].dropna().sort_values()
        colors = ['red' if x < 0 else 'blue' for x in diff_sorted]
        axes[1].barh(diff_sorted.index, diff_sorted.values, color=colors, alpha=0.7)
        axes[1].set_xlabel('Difference (years)')
        axes[1].set_title('Difference Between Data Sources')
        axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
        axes[1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## 4. Creating a Unified Health Index

In [ ]:
# Combine indicators from multiple sources
print("📊 Creating Unified Health Index\n")

# Define indicators to include
indicator_mapping = {
    'Life Expectancy': ['Life Expectancy'],
    'Health Expenditure': ['Health Exp (% GDP)'],
    'Physicians': ['Physicians (per 1000)'],
    'Immunization': ['DTP3 Coverage']
}

unified_data = pd.DataFrame()

# Collect data from all sources
for source_name, source_data in data_sources.items():
    for unified_name, possible_names in indicator_mapping.items():
        for col in possible_names:
            if col in source_data.columns:
                col_name = f"{unified_name}_{source_name.replace(' ', '_')}"
                unified_data[col_name] = source_data[col]
                print(f"✅ {unified_name} from {source_name}")
                break

if not unified_data.empty:
    print(f"\n📊 Unified dataset: {len(unified_data)} countries, {len(unified_data.columns)} indicators")
    display(unified_data.head())

In [ ]:
# Calculate composite health index
if not unified_data.empty:
    # Normalize indicators (0-100 scale)
    normalized_data = pd.DataFrame(index=unified_data.index)
    
    for col in unified_data.columns:
        min_val = unified_data[col].min()
        max_val = unified_data[col].max()
        if max_val > min_val:
            normalized_data[col] = ((unified_data[col] - min_val) / (max_val - min_val)) * 100
    
    # Calculate average for each indicator type
    indicator_averages = pd.DataFrame(index=unified_data.index)
    
    for indicator_type in indicator_mapping.keys():
        type_cols = [c for c in normalized_data.columns if indicator_type in c]
        if type_cols:
            indicator_averages[indicator_type] = normalized_data[type_cols].mean(axis=1)
    
    # Calculate composite index
    indicator_averages['Composite Health Index'] = indicator_averages.mean(axis=1)
    
    print("📊 Composite Health Index:")
    display(indicator_averages.round(2))
    
    # Rank countries
    print("\n🏆 Top 10 Countries by Health Index:")
    top_countries = indicator_averages.sort_values('Composite Health Index', ascending=False).head(10)
    for i, (country, row) in enumerate(top_countries.iterrows(), 1):
        print(f"  {i:2d}. {country}: {row['Composite Health Index']:.1f}")

## 5. Regional Comparison

In [ ]:
# Group countries by region
regions = {
    'Americas': ['BRA', 'USA', 'CAN', 'MEX'],
    'Europe': ['DEU', 'FRA', 'GBR', 'ITA', 'ESP', 'NLD'],
    'Asia': ['CHN', 'IND', 'JPN', 'KOR'],
    'Africa': ['NGA', 'ZAF', 'EGY', 'KEN']
}

# Map country codes
country_to_region = {}
for region, countries in regions.items():
    for country in countries:
        country_to_region[country] = region

# Add region column
if 'indicator_averages' in locals():
    indicator_averages['Region'] = indicator_averages.index.map(country_to_region)
    
    # Regional averages
    regional_avg = indicator_averages.groupby('Region').mean()
    
    print("📊 Regional Averages:")
    display(regional_avg.round(2))
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 6))
    
    regional_avg_sorted = regional_avg.sort_values('Composite Health Index', ascending=True)
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(regional_avg_sorted)))
    
    bars = ax.barh(regional_avg_sorted.index, 
                  regional_avg_sorted['Composite Health Index'],
                  color=colors)
    
    ax.set_xlabel('Composite Health Index (0-100)')
    ax.set_title('Health Index by Region')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 1, bar.get_y() + bar.get_height()/2,
               f'{width:.1f}', ha='left', va='center')
    
    plt.tight_layout()
    plt.show()

## 6. Data Quality Assessment

In [ ]:
# Assess data completeness
print("📊 Data Quality Assessment\n")

if not unified_data.empty:
    # Calculate completeness by source
    completeness = pd.DataFrame({
        'Total Values': len(unified_data),
        'Available': unified_data.count(),
        'Missing': unified_data.isna().sum(),
        'Completeness %': (unified_data.count() / len(unified_data)) * 100
    })
    
    print("📋 Data Completeness by Indicator:")
    display(completeness.round(1))
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    
    completeness_sorted = completeness.sort_values('Completeness %')
    colors = plt.cm.RdYlGn(completeness_sorted['Completeness %'] / 100)
    
    bars = ax.barh(completeness_sorted.index, 
                  completeness_sorted['Completeness %'],
                  color=colors)
    
    ax.set_xlabel('Completeness (%)')
    ax.set_title('Data Completeness by Indicator')
    ax.set_xlim(0, 100)
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add percentage labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 1, bar.get_y() + bar.get_height()/2,
               f'{width:.0f}%', ha='left', va='center')
    
    plt.tight_layout()
    plt.show()

## 7. Export Combined Dataset

In [ ]:
# Export all combined data
import os
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

datasets_to_export = {
    'multi_source_comparison.csv': unified_data if 'unified_data' in locals() else None,
    'composite_health_index.csv': indicator_averages if 'indicator_averages' in locals() else None,
    'regional_averages.csv': regional_avg if 'regional_avg' in locals() else None
}

for filename, data in datasets_to_export.items():
    if data is not None and not data.empty:
        filepath = os.path.join(output_dir, filename)
        data.to_csv(filepath)
        print(f"✅ Saved {filename}")

print(f"\n📁 All combined datasets saved to: {os.path.abspath(output_dir)}/")

## 8. Summary

### Key Findings:

1. **Data Consistency**: High correlation between WHO and World Bank life expectancy data
2. **Regional Disparities**: Significant health index differences between regions
3. **Data Completeness**: Some indicators have missing values for certain countries
4. **Composite Index**: Provides holistic view of health system performance

### Recommendations:

- Use multiple sources for validation
- Account for missing data in analysis
- Consider data collection methodologies when comparing
- Update data regularly for trending analysis

### Next Steps:

- Add more countries and indicators
- Create time series analysis
- Develop predictive models
- Build interactive dashboards

---

**Notebook created:** 2024
**Author:** Epidemiological Datasets Repository